# SHACL Profiling — User Guide

SHACL 1.2 Profiling defines packaging conventions for shapes/data graphs as identifiable resources (`sh:ShapesGraph`/`sh:DataGraph`, `owl:imports`) and vocabulary for declaring which subset of SHACL features something uses. This guide is short because that's genuinely almost the whole story: investigated in full against the live spec text, **no validator runtime behavior gap exists**. Section 2's packaging recommendations are explicitly SHOULD-level metadata — nothing in the spec claims asserting `a sh:ShapesGraph` changes how a validator processes the graph. Sections 2.7 and Annex B of the spec itself are still unfinished `TODO` placeholders in the published Working Draft.

There's exactly one genuinely runtime-relevant piece: an optional inference rule (§5.5) deriving `sh:conformsTo` claims from a validation report. It's covered in section 2 below, along with why it can't fire against `starshacl`'s own report output without extra work from the caller.

## How to run this notebook

1. `pip install "git+https://github.com/hidden-graph/starlayer.git"` (or install the three packages editable from a local checkout — see the root [README](../../README.md)).
2. Run cells top to bottom.

In [1]:
from rdflib.namespace import RDF
from starlayergraph import StarLayerGraph, Namespace, Literal, URIRef
from starshacl import StarShaclValidator

EX = Namespace("http://example.org/")
SH = Namespace("http://www.w3.org/ns/shacl#")

## 1. `sh:ShapesGraph`/`sh:DataGraph` and `owl:imports` — metadata only

Marking a shapes graph `a sh:ShapesGraph` and listing its shapes via `rdfs:member` is purely organizational — it doesn't change what conforms. This passes through `validate()` and the meta-shacl preflight exactly like any other unrecognized-but-well-formed metadata.

In [2]:
data = StarLayerGraph()
data.parse(data="""
    @prefix ex: <http://example.org/> .
    ex:alice a ex:Person ; ex:age 30 .
""", format="turtle")

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

    ex:MyShapesGraph a sh:ShapesGraph ;
      rdfs:member ex:PersonShape .

    ex:PersonShape a sh:NodeShape ;
      sh:targetClass ex:Person ;
      sh:property [ sh:path ex:age ; sh:datatype xsd:integer ] .
""", format="turtle")

result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("conforms:", result.conforms)

meta_result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=True)
print("meta_shacl conforms:", meta_result.conforms)

conforms: True
meta_shacl conforms: True


`owl:imports` is separately handled as real plumbing, not just pass-through metadata: `validate()` resolves a shapes graph's `owl:imports` chain (per the spec's `^owl:versionIRI?/owl:imports` algorithm) before validation runs, via a caller-supplied loader function — this library doesn't fetch anything over the network itself.

## 2. `sh:usedDataGraph`/`sh:usedShapesGraph` — report provenance

Separately from Profiling itself (this is SHACL 1.2 Core §6.7.1.5-6.7.1.6), a validation report can optionally record which data/shapes graph it was run against. `rdflib.Graph` objects are normally anonymous in memory, so there's no default IRI to use — pass one explicitly via `data_graph_iri=`/`shapes_graph_iri=` and it's added to the report; omit them (the default) and nothing is added, no invented IRI.

In [3]:
data = StarLayerGraph()
data.add((EX.alice, EX.age, Literal(30)))

shapes = StarLayerGraph()
shapes.parse(data="""
    @prefix ex: <http://example.org/> .
    @prefix sh: <http://www.w3.org/ns/shacl#> .
    @prefix xsd: <http://www.w3.org/2001/XMLSchema#> .
    ex:S a sh:NodeShape ;
      sh:targetNode ex:alice ;
      sh:property [ sh:path ex:age ; sh:datatype xsd:integer ] .
""", format="turtle")

# absent by default
default_result = StarShaclValidator().validate(data_graph=data, shacl_graph=shapes, meta_shacl=False)
print("usedDataGraph present by default:", len(list(default_result.report_graph.objects(None, SH.usedDataGraph))) > 0)

# present once the caller supplies real IRI identity for each graph
result = StarShaclValidator().validate(
    data_graph=data, shacl_graph=shapes, meta_shacl=False,
    data_graph_iri="http://example.org/graphs/data1",
    shapes_graph_iri="http://example.org/graphs/shapes1",
)
print("conforms:", result.conforms)
print("usedDataGraph:", list(result.report_graph.objects(None, SH.usedDataGraph)))
print("usedShapesGraph:", list(result.report_graph.objects(None, SH.usedShapesGraph)))

usedDataGraph present by default: False
conforms: True
usedDataGraph: [rdflib.term.URIRef('http://example.org/graphs/data1')]
usedShapesGraph: [rdflib.term.URIRef('http://example.org/graphs/shapes1')]


## 3. The `sh:conformsTo` inference rule

Profiling §5.5 defines one optional (MAY-level) inference rule, mechanically just a plain SPARQL CONSTRUCT — the same shape of rule `starshacl`'s existing `sh:SPARQLRule` engine already executes. So there's no missing *capability*. But it can't fire against `starshacl`'s own report output as-is: the rule needs `?x sh:validationReport ?y`, linking a data-graph *resource* to the report, and nothing in pySHACL's report ever produces that triple (confirmed by grepping its entire codebase — zero references to `sh:validationReport`).

In [4]:
spec_rule = """
PREFIX sh: <http://www.w3.org/ns/shacl#>
CONSTRUCT { ?x sh:conformsTo ?z }
WHERE { ?x sh:validationReport ?y . ?y sh:usedShapesGraph ?z ; sh:conforms true . }
"""

# run the spec's exact rule verbatim against starshacl's own report - it can't fire,
# because nothing links a data-graph resource to the report via sh:validationReport
print("fires as-is:", len(list(result.report_graph.query(spec_rule))) > 0)
print("sh:validationReport triples present:", len(list(result.report_graph.triples((None, SH.validationReport, None)))))

fires as-is: False
sh:validationReport triples present: 0


In [5]:
# supplying that one missing link is an application-level modeling decision (what IRI
# names "the data graph" as a resource?) - once supplied, the rule fires exactly as specified
report_node = next(result.report_graph.subjects(RDF.type, SH.ValidationReport))

linked = StarLayerGraph()
for t in result.report_graph:
    linked.add(t)
data_graph_resource = URIRef("http://example.org/graphs/data1")
linked.add((data_graph_resource, SH.validationReport, report_node))

derived = list(linked.query(spec_rule))
print("fires once the caller supplies the sh:validationReport link:", len(derived) > 0)
for row in derived:
    print(row)

fires once the caller supplies the sh:validationReport link: True
(rdflib.term.URIRef('http://example.org/graphs/data1'), rdflib.term.URIRef('http://www.w3.org/ns/shacl#conformsTo'), rdflib.term.URIRef('http://example.org/graphs/shapes1'))


## Further work

- **A `sh:conformsTo` convenience helper** — a small function taking a report graph plus the data/shapes graph IRIs and returning/asserting the derived triple (section 3 above, done by hand) — is deliberately not built yet. It needs a caller-supplied IRI-naming convention for their data/shapes graphs that doesn't exist generically; revisit when a concrete caller actually wants to record `sh:conformsTo` claims against a real dataset.